# Inundation regime analysis — Hydraulic bifurcation in HEC-RAS

Case study: **Besaya River — Corrales de Buelna**

The HEC-RAS flooded area distribution is **bimodal**: two groups of simulations
converge to different solutions despite using the same Manning values.
This notebook identifies the physical cause: a **topographic threshold** (topographic
saddle at ~60 m a.s.l.) that, when the water surface exceeds it, floods a secondary
zone of ~7.4 ha.

**Analysis workflow:**
1. Load results and classify regimes (2-component GMM)
2. Verify that Manning does **not** predict the regime (t-test / Spearman)
3. Locate the zone exclusive to the HIGH regime
4. Identify the topographic saddle (minimum-elevation point on the barrier between zones)
5. Publication figure — hydraulic bifurcation

In [ ]:
from pathlib import Path

import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rioxarray as rxr
import xarray as xr
from scipy import ndimage, stats
from sklearn.mixture import GaussianMixture

from pyhydra.modeling.hydraulic.sensitivity import load_flood_ensemble


## Paths — adjust for your environment

In [ ]:
ZENODO_DIR   = Path("..").resolve()
HECRAS_DIR   = ZENODO_DIR / "simulations" / "HEC-RAS"
DEM_PATH     = ZENODO_DIR / "models" / "SFINCS" / "gis" / "dep.tif"

CRS        = "EPSG:25830"
THRESHOLD  = 0.05          # calado mínimo para celda mojada (m)
CELL_AREA  = 25.0          # m² por celda (5 m × 5 m)
REMOVE_SIMS = [29, 295, 633, 724, 755]   # no-convergencias eliminadas en nb04

## 1 · Load results and classify regimes (GMM)

A 2-component Gaussian Mixture Model is fitted to the HEC-RAS flooded area.
The regime boundary is determined by the model, not set manually.

In [ ]:
hr = pd.read_csv(ZENODO_DIR / "data" / "hecras_sensitivity_results.csv", index_col=0).drop(index=REMOVE_SIMS)
sf = pd.read_csv(ZENODO_DIR / "data" / "sfincs_sensitivity_results.csv", index_col=0).drop(index=REMOVE_SIMS)

gmm = GaussianMixture(n_components=2, random_state=42, n_init=10)
gmm.fit(hr["flooded_area_km2"].values.reshape(-1, 1))
labels = gmm.predict(hr["flooded_area_km2"].values.reshape(-1, 1))

# Label 0 = low-area regime
if gmm.means_.flatten()[0] > gmm.means_.flatten()[1]:
    labels = 1 - labels
hr["regime"] = labels

r0_idx = hr[hr["regime"] == 0].index.tolist()
r1_idx = hr[hr["regime"] == 1].index.tolist()

print(f"LOW  regime (label=0): n={len(r0_idx):4d}  area={hr.loc[r0_idx,'flooded_area_km2'].mean():.4f} km²")
print(f"HIGH regime (label=1): n={len(r1_idx):4d}  area={hr.loc[r1_idx,'flooded_area_km2'].mean():.4f} km²")
print(f"\nMedias GMM:  {sorted(gmm.means_.flatten().round(4))}")
print(f"Stds  GMM:   {sorted(np.sqrt(gmm.covariances_.flatten()).round(4))}")


## 2 · Manning does **not** predict the regime (t-test and Spearman)

If the two regimes were caused by Manning values, some coefficient
should show statistically significant differences between groups.

In [ ]:
comb = pd.read_csv(
    ZENODO_DIR / "data" / "monte_carlo_combinations.csv"
).drop(index=REMOVE_SIMS).reset_index(drop=False)

land_uses = ["Trees", "Dense vegetation", "Urban vegetation", "Infrastructure",
             "Sparse vegetation", "Residential", "Industrial", "River", "Brushland"]

rows = []
mask0 = comb["index"].isin(r0_idx)
mask1 = comb["index"].isin(r1_idx)
for col in land_uses:
    t, p_t  = stats.ttest_ind(comb[mask0][col], comb[mask1][col])
    rho, p_r = stats.spearmanr(comb[col], hr["regime"].values)
    rows.append({"Uso del suelo": col,
                 "Media BAJO": comb[mask0][col].mean(),
                 "Media ALTO": comb[mask1][col].mean(),
                 "t": t, "p (t-test)": p_t,
                 "ρ (Spearman)": rho, "p (Spearman)": p_r})

tab = pd.DataFrame(rows).set_index("Uso del suelo")
print("No coefficient is significant → Manning does NOT separate the regimes")
tab.round(4)


## 3 · SFINCS area does correlate with the HEC-RAS regime

Although individual Manning values do not predict the regime, the integrated
SFINCS response (which captures the total energy balance) does. This confirms
that the regime depends on flow energy, not on any single land-use class.

In [ ]:
rho_sf, p_sf = stats.spearmanr(sf["flooded_area_km2"], hr["regime"])
rho_mn, p_mn = stats.spearmanr(hr["manning_mean"], hr["regime"])
print(f"SFINCS area   vs HR regime:  ρ={rho_sf:+.3f}  p={p_sf:.2e}")
print(f"HR manning_mean vs regime:   ρ={rho_mn:+.3f}  p={p_mn:.2e}")

fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(sf.loc[r0_idx, "flooded_area_km2"], hr.loc[r0_idx, "flooded_area_km2"],
           alpha=0.5, s=15, color="steelblue", label=f"Régimen BAJO (n={len(r0_idx)})")
ax.scatter(sf.loc[r1_idx, "flooded_area_km2"], hr.loc[r1_idx, "flooded_area_km2"],
           alpha=0.25, s=8, color="tomato", label=f"Régimen ALTO (n={len(r1_idx)})")
ax.set_xlabel("SFINCS área inundada (km²)")
ax.set_ylabel("HEC-RAS área inundada (km²)")
ax.set_title(f"SFINCS predice el régimen HEC-RAS  (ρ={rho_sf:+.3f}, p={p_sf:.0e})")
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()


## 4 · Locate the zone exclusive to the HIGH regime and the topographic saddle

Inundation frequency maps are built for each regime.  
The **zone exclusive to the HIGH regime** is the spatial difference between both maps.  
The **topographic saddle** is the minimum-elevation point on the corridor between the
permanently flooded zone (LOW regime) and the exclusive zone.

In [ ]:
def build_freq_map(sim_list, n_sample=60, seed=0):
    """Mean inundation frequency raster from a sample of simulations."""
    rng = np.random.default_rng(seed)
    sample = rng.choice(sim_list, min(n_sample, len(sim_list)), replace=False)
    ref_path = HECRAS_DIR / f"hamax_sim_{sim_list[0]}.tif"
    ref = rxr.open_rasterio(ref_path, masked=True).squeeze("band", drop=True).rio.write_crs(CRS)
    acc = xr.zeros_like(ref, dtype=float)
    for s in sorted(sample):
        f = HECRAS_DIR / f"hamax_sim_{s}.tif"
        if not f.exists():
            continue
        da = rxr.open_rasterio(f, masked=True).squeeze("band", drop=True).rio.write_crs(CRS)
        acc += (da >= THRESHOLD).fillna(0).astype(float)
    return acc / min(n_sample, len(sim_list))

print("Construyendo mapas de frecuencia...")
freq_bajo = build_freq_map(r0_idx, n_sample=len(r0_idx))   # use all 97
freq_alto = build_freq_map(r1_idx, n_sample=80)

# Zone exclusively wet in HIGH regime
threshold_zone = (freq_alto > 0.6) & (freq_bajo < 0.2)
n_cells = int(threshold_zone.sum())
print(f"Zone exclusive to HIGH regime: {n_cells} cells = {n_cells*CELL_AREA*1e-4:.1f} ha")


In [ ]:
# Load DEM reprojected to HEC-RAS grid
dem = rxr.open_rasterio(str(DEM_PATH), masked=True).squeeze().rio.write_crs(CRS)
dem_repr = dem.rio.reproject_match(freq_bajo)

# Always-wet zone in LOW regime (core flood area)
ref = rxr.open_rasterio(HECRAS_DIR / f"hamax_sim_{r0_idx[0]}.tif", masked=True).squeeze("band", drop=True).rio.write_crs(CRS)
always_bajo = xr.ones_like(ref, dtype=float)
for s in r0_idx:
    f = HECRAS_DIR / f"hamax_sim_{s}.tif"
    if not f.exists():
        continue
    da = rxr.open_rasterio(f, masked=True).squeeze("band", drop=True).rio.write_crs(CRS)
    always_bajo *= (da >= THRESHOLD).fillna(0).astype(float)

# Saddle point: minimum elevation on the barrier between the two zones
always_np = always_bajo.values > 0
thresh_np = threshold_zone.values.astype(bool)
dem_np    = dem_repr.values

dil_thresh = ndimage.binary_dilation(thresh_np,  iterations=5)
dil_always = ndimage.binary_dilation(always_np,  iterations=5)
corridor   = dil_thresh & dil_always & (~always_np) & (~thresh_np)

elev_corr = dem_np[corridor]
elev_corr = elev_corr[~np.isnan(elev_corr)]
corr_rows, corr_cols = np.where(corridor)
min_idx   = np.nanargmin(dem_np[corridor])
SADDLE_X  = float(dem_repr.x.values[corr_cols[min_idx]])
SADDLE_Y  = float(dem_repr.y.values[corr_rows[min_idx]])
SADDLE_Z  = float(dem_np[corr_rows[min_idx], corr_cols[min_idx]])

print(f"Topographic saddle (minimum of the barrier):")
print(f"  Elevation: {SADDLE_Z:.2f} m a.s.l.")
print(f"  X = {SADDLE_X:.0f},  Y = {SADDLE_Y:.0f}  (EPSG:25830)")
print(f"\nOverflow threshold towards the secondary zone: ~{SADDLE_Z:.1f} m.")

# Elevation stats of threshold zone
elev_thresh = dem_repr.where(threshold_zone).values.ravel()
elev_thresh = elev_thresh[~np.isnan(elev_thresh)]
print(f"\nElevation of the zone exclusive to HIGH regime (P5–P95): "
      f"{np.percentile(elev_thresh,5):.1f} – {np.percentile(elev_thresh,95):.1f} m")


## 5 · Publication figure — Hydraulic bifurcation

In [ ]:
# Crop extent to floodplain (DEM < 85 m with any flooding)
flood_any = freq_alto > 0.05
ys, xs = np.where(flood_any.values & (dem_repr.values < 85))
x_coords = dem_repr.x.values[xs]; y_coords = dem_repr.y.values[ys]
xlim = [x_coords.min() - 200, x_coords.max() + 200]
ylim = [y_coords.min() - 200, y_coords.max() + 200]
y2d, x2d = np.meshgrid(freq_bajo.y.values, freq_bajo.x.values, indexing="ij")

fig = plt.figure(figsize=(18, 9))
gs  = fig.add_gridspec(2, 3, hspace=0.4, wspace=0.35)

def _map_panel(ax, freq, title, cmap, label_cbar):
    dem_repr.plot(ax=ax, cmap="Greys_r", alpha=0.55, vmin=55, vmax=85, add_colorbar=False)
    freq.where(freq > 0.05).plot(ax=ax, cmap=cmap, alpha=0.8, vmin=0, vmax=1,
        cbar_kwargs={"label": label_cbar, "shrink": 0.62, "pad": 0.02})
    ax.contour(x2d, y2d, threshold_zone.values.astype(float),
               levels=[0.5], colors="limegreen", linewidths=2, linestyles="--")
    ax.plot(SADDLE_X, SADDLE_Y, "r^", ms=13, zorder=7,
            label=f"Silla {SADDLE_Z:.1f} m")
    ax.set_xlim(xlim); ax.set_ylim(ylim)
    ax.set_title(title, fontsize=10, fontweight="bold")
    ax.set_xlabel("X UTM (m)")

ax0 = fig.add_subplot(gs[:, 0])
_map_panel(ax0, freq_bajo,
           f"Régimen BAJO (n={len(r0_idx)})\nÁrea media ≈ {hr.loc[r0_idx,'flooded_area_km2'].mean():.3f} km²",
           "Blues", "Frec. inundación (BAJO)")
ax0.set_ylabel("Y UTM (m)")
ax0.legend(fontsize=8)

ax1 = fig.add_subplot(gs[:, 1])
_map_panel(ax1, freq_alto,
           f"Régimen ALTO (n={len(r1_idx)})\nÁrea media ≈ {hr.loc[r1_idx,'flooded_area_km2'].mean():.3f} km²",
           "Reds", "Frec. inundación (ALTO)")
ax1.set_ylabel("")
green_patch = mpatches.Patch(edgecolor="limegreen", facecolor="none",
                              ls="--", lw=2, label="Zona secundaria (~7.4 ha)")
ax1.legend(handles=list(ax1.get_lines()) + [green_patch], fontsize=8)

# Distribution panel
ax2 = fig.add_subplot(gs[0, 2])
ax2.hist(hr.loc[r0_idx, "flooded_area_km2"], bins=20, color="steelblue",
         alpha=0.75, label=f"Régimen BAJO (n={len(r0_idx)})")
ax2.hist(hr.loc[r1_idx, "flooded_area_km2"], bins=35, color="tomato",
         alpha=0.6,  label=f"Régimen ALTO (n={len(r1_idx)})")
ax2.hist(sf["flooded_area_km2"], bins=35, color="forestgreen",
         alpha=0.5,  label="SFINCS (unimodal)")
ax2.set_xlabel("Área inundada (km²)"); ax2.set_ylabel("N simulaciones")
ax2.set_title("Distribución del área inundada\nHEC-RAS bimodal / SFINCS unimodal", fontsize=10)
ax2.legend(fontsize=8); ax2.grid(True, alpha=0.3)

# SFINCS predictor panel
ax3 = fig.add_subplot(gs[1, 2])
ax3.scatter(sf.loc[r0_idx, "flooded_area_km2"], hr.loc[r0_idx, "flooded_area_km2"],
            alpha=0.55, s=12, color="steelblue", label="Régimen BAJO")
ax3.scatter(sf.loc[r1_idx, "flooded_area_km2"], hr.loc[r1_idx, "flooded_area_km2"],
            alpha=0.2,  s=8,  color="tomato",    label="Régimen ALTO")
rho_sf, _ = stats.spearmanr(sf["flooded_area_km2"], hr["regime"])
ax3.set_xlabel("SFINCS área (km²)"); ax3.set_ylabel("HEC-RAS área (km²)")
ax3.set_title(f"SFINCS como predictor del régimen HR\n(ρ={rho_sf:+.3f}, p<10⁻³¹)", fontsize=10)
ax3.legend(fontsize=8); ax3.grid(True, alpha=0.3)

plt.suptitle(
    "Bifurcación hidráulica — Río Besaya, Corrales de Buelna\n"
    f"Umbral topográfico: silla en X={SADDLE_X:.0f}, Y={SADDLE_Y:.0f}, z={SADDLE_Z:.1f} m s.n.m.",
    fontsize=12, y=1.01
)
plt.savefig("hydraulic_bifurcation.png", dpi=150, bbox_inches="tight")
plt.show()
print("Guardado: hydraulic_bifurcation.png")


## 6 · Transect elevation profile across the barrier

The elevation histogram of the corridor between zones shows the elevation
range of the topographic barrier.

In [ ]:
elev_thresh_zone = dem_repr.where(threshold_zone).values.ravel()
elev_thresh_zone = elev_thresh_zone[~np.isnan(elev_thresh_zone)]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
ax.hist(elev_corr, bins=40, color="sienna", edgecolor="white", lw=0.3, alpha=0.8)
ax.axvline(SADDLE_Z, color="red", lw=2, label=f"Silla = {SADDLE_Z:.2f} m")
ax.axvline(np.percentile(elev_corr, 10), color="orange", lw=1.5, ls="--",
           label=f"P10 = {np.percentile(elev_corr,10):.1f} m")
ax.set_xlabel("Elevación (m)"); ax.set_ylabel("N celdas")
ax.set_title("Elevación en el corredor (barrera)")
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

ax = axes[1]
ax.hist(elev_thresh_zone, bins=40, color="tomato", edgecolor="white", lw=0.3, alpha=0.8)
ax.axvline(np.median(elev_thresh_zone), color="red", lw=2,
           label=f"Mediana = {np.median(elev_thresh_zone):.1f} m")
ax.set_xlabel("Elevación (m)"); ax.set_ylabel("N celdas")
ax.set_title("Elevación de la zona exclusiva del régimen ALTO")
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

plt.suptitle("Perfil de elevación — barrera topográfica y zona secundaria", fontsize=11)
plt.tight_layout()
plt.show()

print(f"Resumen barrera:")
print(f"  Saddle (barrier minimum): {SADDLE_Z:.2f} m")
print(f"  P10 corredor:                 {np.percentile(elev_corr,10):.2f} m")
print(f"  P90 corredor:                 {np.percentile(elev_corr,90):.2f} m")
print(f"  Zona secundaria mediana:      {np.median(elev_thresh_zone):.2f} m")


## 7 · Interpretation and conclusions

### Why two regimes?

A **topographic barrier** exists between the main Besaya channel and a secondary
lateral floodplain. The lowest point on this barrier (the saddle) lies at
**~60 m a.s.l.**

- When the model water surface reaches or exceeds that elevation, flow overtops
  into the secondary zone (+7.4 ha) → **HIGH regime** (~700 m²).
- When the water surface does not reach that threshold, flooding is confined to
  the main channel → **LOW regime** (~577 m²).

### Why does HEC-RAS capture this and SFINCS does not?

| Model | Equations | Response to threshold |
|--------|-----------|---------------------|
| **HEC-RAS** | Full 2D Saint-Venant (with advection) | Discontinuous — bimodal |
| **SFINCS** | Simplified inertial (no advection) | Continuous — unimodal |

The advective terms ρu·∇u allow HEC-RAS to transfer hydraulic momentum
towards the saddle point. Near the threshold, small energy variations (driven
by roughness, depth, or upstream velocity) determine whether flow overtops the
barrier. The response is **discontinuous**.

SFINCS, without those terms, does not accumulate momentum towards the saddle → the
response is smooth and the transition does not produce bimodality.

### Is this a numerical artefact?

No. The correlation between SFINCS area and the HEC-RAS regime (ρ=+0.36, p<10⁻³¹)
confirms a real physical signal. Individual Manning values do not predict the regime
(all p>0.35) because the threshold is sensitive to **integrated flow energy**, not
to any specific land-use class.

### Implication for the paper

> The bimodal flooded area distribution in HEC-RAS is not an analysis artefact
> but a consequence of the correct representation of a real topographic threshold
> through full Saint-Venant equations. SFINCS cannot reproduce it by design.
> This adds a third source of uncertainty — the **structure of the response
> distribution** — that only emerges when comparing models with different
> governing equation formulations.